<a href="https://colab.research.google.com/github/zaidbinnaveed/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is a classification task. The goal is to predict whether a piece of content is declining and therefore needs a refresh. The FlyRank starter data already contains is_declining_label, a binary flag assigned to each content item. This matches the "will this one decline or recover" pattern from the framing guide, which maps to classification rather than ranking (since I am not ordering items relative to each other) or clustering (since I have a predefined label rather than unknown groups to discover).

## 2. Target or proxy

The target is is_declining_label, a binary value (0 or 1). This label is derived from trend_direction, which is itself computed from trend_pct, an observed trailing 90 day trend rather than a rule invented for this assignment. That makes it a legitimate prediction target. However, trend_direction and trend_pct can never be used as input features, since including them would leak the answer directly into the model. Instead, the feature set will draw from signals available before the trend outcome is known, such as ctr, engagement_rate, scroll_rate, avg_position, word_count, content_type, and ai_traffic_pct.

## 3. Success metric

Declining content is likely a minority class, so plain accuracy would be misleading. A model that always predicts "not declining" could still score well on accuracy alone. I will use ROC AUC along with precision and recall compared against the base rate of is_declining_label. This aligns with the recommendation in the framing guide for this task type. I will confirm the actual class balance in the data before finalizing this choice.

## 4. The unit of analysis, as a real dataframe

One row represents one pseudonymized content item belonging to one of 32 clients, summarizing its trailing 90 day performance metrics. Both content_id and client_id are pseudonyms. They are useful for grouping and splitting the data, for example performing a grouped train and test split by client_id to avoid leakage across clients, but they should never be used as model features.

In [ ]:
!git clone https://github.com/zaidbinnaveed/FlyRank-ML-Internship.git

Cloning into 'FlyRank-ML-Internship'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 114 (delta 31), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 1.85 MiB | 8.05 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [ ]:
import os
os.chdir("FlyRank-ML-Internship")
print(os.getcwd())

/content/FlyRank-ML-Internship/FlyRank-ML-Internship


In [ ]:
!ls data/raw/

content_refresh_anonymized.csv


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df['is_declining_label'].value_counts(normalize=True)

,proportion
is_declining_label,
1,0.542067
0,0.457933


## 5. Why ML beats a fixed rule here

A simple rule such as "flag as declining if CTR drops below a fixed threshold" would not hold up in practice. Decline shows up differently depending on content_type, since missingness patterns alone vary substantially by type, and also depending on each client's baseline behavior. A drop in scroll_rate combined with a rise in ai_traffic_pct can mean something entirely different than the same drop occurring in isolation. A single threshold cannot account for these interactions across 32 clients with different starting points. A machine learning model can weigh multiple signals together and adapt to these patterns, whereas a fixed rule would require constant retuning per client and would still miss combinations that were never anticipated.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.